# 📚 Business Glossary & Policy Engine — Enterprise Data Governance

**Self-contained business glossary management with automated policy enforcement across all datasets**

## Business Context

### The Problem
- **Undocumented data** — Tables/columns lack business definitions
- **No DDL comments** — Schema changes without documentation
- **Inconsistent terminology** — Same concept, different names
- **No governance** — No enforcement of metadata standards
- **Manual processes** — Manual tracking of business terms

### The Solution
**Enterprise business glossary management + Automated policy engine**

## Capabilities

### 📚 Business Glossary (Self-Contained)
- ✅ **Term management** — Create, update, approve, deprecate terms
- ✅ **Hierarchical organization** — Categories, domains, sub-categories
- ✅ **Linkage** — Map terms to physical tables/columns
- ✅ **Versioning** — Full history of definition changes
- ✅ **Stewardship** — Assign owners and stewards
- ✅ **Search** — Full-text search across all terms
- ✅ **Approval workflows** — Pending → Approved → Published

### 🛡️ Policy Engine (Automated Enforcement)
- ✅ **DDL comment validation** — Flag tables without comments
- ✅ **Business definition validation** — Flag unmapped columns
- ✅ **Naming convention checks** — Enforce standards
- ✅ **Auto-scanning** — Run on every dataset (scheduled)
- ✅ **Violation tracking** — Delta table audit log
- ✅ **Auto-remediation** — Suggest fixes
- ✅ **Compliance reporting** — Executive dashboards

### 🎯 No Purview Required!
All metadata managed in Fabric Delta tables with full lineage and governance.

---

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  Load Fabric Utilities                                              ║
# ╚══════════════════════════════════════════════════════════════════════╝
%run 00_fabric_native_utils

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 1: Business Glossary Schema & Infrastructure               ║
# ╚══════════════════════════════════════════════════════════════════════╝

from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
import json
from typing import Dict, List, Optional

print("📚 Business Glossary & Policy Engine")
print("=" * 80)

# === Create Business Glossary Tables ===

# 1. Glossary Terms (Core)
spark.sql("""
    CREATE TABLE IF NOT EXISTS metadata.glossary_terms (
        term_id STRING,
        term_name STRING,
        display_name STRING,
        definition STRING,
        business_definition STRING,
        technical_definition STRING,
        examples STRING,
        category STRING,
        domain STRING,
        status STRING,  -- draft, pending_review, approved, published, deprecated
        version INT,
        created_by STRING,
        created_timestamp TIMESTAMP,
        updated_by STRING,
        updated_timestamp TIMESTAMP,
        approved_by STRING,
        approved_timestamp TIMESTAMP,
        steward STRING,
        owner STRING,
        tags ARRAY<STRING>,
        related_terms ARRAY<STRING>,
        synonyms ARRAY<STRING>,
        acronyms ARRAY<STRING>
    ) USING DELTA
    PARTITIONED BY (domain, status)
""")

# 2. Term Versions (History)
spark.sql("""
    CREATE TABLE IF NOT EXISTS metadata.glossary_term_versions (
        version_id STRING,
        term_id STRING,
        version INT,
        definition STRING,
        changed_by STRING,
        changed_timestamp TIMESTAMP,
        change_reason STRING,
        previous_version INT
    ) USING DELTA
    PARTITIONED BY (DATE(changed_timestamp))
""")

# 3. Term Linkages (Physical Mapping)
spark.sql("""
    CREATE TABLE IF NOT EXISTS metadata.glossary_term_linkages (
        linkage_id STRING,
        term_id STRING,
        term_name STRING,
        object_type STRING,  -- table, column, report, dashboard
        database_name STRING,
        schema_name STRING,
        table_name STRING,
        column_name STRING,
        full_path STRING,
        confidence_score DOUBLE,  -- 0.0-1.0 (manual=1.0, auto=<1.0)
        linkage_type STRING,  -- manual, auto_suggested, auto_approved
        linked_by STRING,
        linked_timestamp TIMESTAMP,
        verified BOOLEAN,
        verified_by STRING,
        verified_timestamp TIMESTAMP
    ) USING DELTA
    PARTITIONED BY (database_name)
""")

# 4. Glossary Categories
spark.sql("""
    CREATE TABLE IF NOT EXISTS metadata.glossary_categories (
        category_id STRING,
        category_name STRING,
        parent_category STRING,
        description STRING,
        domain STRING,
        created_by STRING,
        created_timestamp TIMESTAMP
    ) USING DELTA
""")

# 5. Policy Definitions
spark.sql("""
    CREATE TABLE IF NOT EXISTS metadata.governance_policies (
        policy_id STRING,
        policy_name STRING,
        policy_type STRING,  -- ddl_comment, business_definition, naming_convention, data_quality
        description STRING,
        severity STRING,  -- critical, high, medium, low
        enabled BOOLEAN,
        auto_remediate BOOLEAN,
        scope STRING,  -- all, database, table_pattern
        scope_filter STRING,
        validation_query STRING,
        created_by STRING,
        created_timestamp TIMESTAMP,
        last_run_timestamp TIMESTAMP
    ) USING DELTA
""")

# 6. Policy Violations
spark.sql("""
    CREATE TABLE IF NOT EXISTS metadata.governance_policy_violations (
        violation_id STRING,
        policy_id STRING,
        policy_name STRING,
        severity STRING,
        object_type STRING,
        database_name STRING,
        schema_name STRING,
        table_name STRING,
        column_name STRING,
        full_path STRING,
        violation_details STRING,
        detected_timestamp TIMESTAMP,
        status STRING,  -- new, acknowledged, remediated, waived
        remediation_suggestion STRING,
        remediated_by STRING,
        remediated_timestamp TIMESTAMP,
        waiver_reason STRING,
        waived_by STRING
    ) USING DELTA
    PARTITIONED BY (DATE(detected_timestamp), severity)
""")

# 7. Metadata Compliance Scorecard
spark.sql("""
    CREATE TABLE IF NOT EXISTS metadata.metadata_compliance_scorecard (
        scorecard_id STRING,
        database_name STRING,
        table_name STRING,
        total_columns INT,
        columns_with_comments INT,
        columns_with_business_definitions INT,
        columns_with_glossary_terms INT,
        compliance_score DOUBLE,
        last_scanned_timestamp TIMESTAMP,
        violations_critical INT,
        violations_high INT,
        violations_medium INT,
        violations_low INT
    ) USING DELTA
    PARTITIONED BY (database_name)
""")

print("\n✅ Business glossary tables initialized")
print("=" * 80)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 2: Business Glossary Manager                               ║
# ╚══════════════════════════════════════════════════════════════════════╝

class BusinessGlossaryManager:
    """
    Manage business glossary terms with full lifecycle support.
    Self-contained - no Purview dependency!
    """
    
    def create_term(
        self,
        term_name: str,
        definition: str,
        category: str,
        domain: str,
        business_definition: str = None,
        technical_definition: str = None,
        steward: str = None,
        tags: List[str] = None
    ) -> str:
        """
        Create a new business glossary term.
        
        Args:
            term_name: Unique term identifier (e.g., "premium_amount")
            definition: Main definition
            category: Category (e.g., "Financial", "Customer", "Claims")
            domain: Business domain (e.g., "Policy", "Claims", "Finance")
            business_definition: Business user-friendly definition
            technical_definition: Technical/IT definition
            steward: Data steward email
            tags: List of tags
            
        Returns:
            term_id
        """
        from notebookutils import mssparkutils
        
        term_id = f"TERM_{datetime.now().strftime('%Y%m%d%H%M%S')}_{hash(term_name) % 10000}"
        current_user = notebookutils.runtime.context.get("userEmail", "system")
        
        term_record = {
            "term_id": term_id,
            "term_name": term_name,
            "display_name": term_name.replace("_", " ").title(),
            "definition": definition,
            "business_definition": business_definition or definition,
            "technical_definition": technical_definition,
            "examples": None,
            "category": category,
            "domain": domain,
            "status": "draft",
            "version": 1,
            "created_by": current_user,
            "created_timestamp": datetime.now(),
            "updated_by": current_user,
            "updated_timestamp": datetime.now(),
            "approved_by": None,
            "approved_timestamp": None,
            "steward": steward or current_user,
            "owner": current_user,
            "tags": tags or [],
            "related_terms": [],
            "synonyms": [],
            "acronyms": []
        }
        
        # Save to Delta table
        spark.createDataFrame([term_record]).write.format("delta") \
            .mode("append") \
            .saveAsTable("metadata.glossary_terms")
        
        print(f"✅ Term created: {term_name} (ID: {term_id})")
        return term_id
    
    def approve_term(self, term_id: str, approver: str = None) -> bool:
        """Approve a term (moves to published status)."""
        from notebookutils import mssparkutils
        
        approver = approver or mssparkutils.runtime.context.get("userEmail", "system")
        
        # Update status
        spark.sql(f"""
            UPDATE metadata.glossary_terms
            SET status = 'published',
                approved_by = '{approver}',
                approved_timestamp = CURRENT_TIMESTAMP,
                updated_timestamp = CURRENT_TIMESTAMP
            WHERE term_id = '{term_id}'
        """)
        
        print(f"✅ Term approved: {term_id}")
        return True
    
    def link_term_to_column(
        self,
        term_id: str,
        database_name: str,
        table_name: str,
        column_name: str,
        confidence: float = 1.0,
        linkage_type: str = "manual"
    ) -> str:
        """
        Link a glossary term to a physical column.
        
        Args:
            term_id: Glossary term ID
            database_name: Database name
            table_name: Table name
            column_name: Column name
            confidence: Confidence score (0.0-1.0)
            linkage_type: manual, auto_suggested, auto_approved
            
        Returns:
            linkage_id
        """
        from notebookutils import mssparkutils
        
        # Get term name
        term = spark.sql(f"""
            SELECT term_name FROM metadata.glossary_terms WHERE term_id = '{term_id}'
        """).collect()
        
        if not term:
            print(f"❌ Term not found: {term_id}")
            return None
        
        term_name = term[0]["term_name"]
        
        linkage_id = f"LINK_{datetime.now().strftime('%Y%m%d%H%M%S')}_{hash(column_name) % 10000}"
        current_user = mssparkutils.runtime.context.get("userEmail", "system")
        
        linkage_record = {
            "linkage_id": linkage_id,
            "term_id": term_id,
            "term_name": term_name,
            "object_type": "column",
            "database_name": database_name,
            "schema_name": "dbo",
            "table_name": table_name,
            "column_name": column_name,
            "full_path": f"{database_name}.{table_name}.{column_name}",
            "confidence_score": confidence,
            "linkage_type": linkage_type,
            "linked_by": current_user,
            "linked_timestamp": datetime.now(),
            "verified": linkage_type == "manual",
            "verified_by": current_user if linkage_type == "manual" else None,
            "verified_timestamp": datetime.now() if linkage_type == "manual" else None
        }
        
        spark.createDataFrame([linkage_record]).write.format("delta") \
            .mode("append") \
            .saveAsTable("metadata.glossary_term_linkages")
        
        print(f"✅ Term linked: {term_name} → {database_name}.{table_name}.{column_name}")
        return linkage_id
    
    def search_terms(self, keyword: str) -> DataFrame:
        """Search glossary terms by keyword."""
        
        results = spark.sql(f"""
            SELECT 
                term_id,
                term_name,
                display_name,
                definition,
                business_definition,
                category,
                domain,
                status,
                steward
            FROM metadata.glossary_terms
            WHERE status = 'published'
              AND (
                  LOWER(term_name) LIKE '%{keyword.lower()}%'
                  OR LOWER(definition) LIKE '%{keyword.lower()}%'
                  OR LOWER(business_definition) LIKE '%{keyword.lower()}%'
                  OR array_contains(tags, '{keyword}')
              )
            ORDER BY term_name
        """)
        
        return results
    
    def get_term_lineage(self, term_id: str) -> DataFrame:
        """Get all physical objects linked to a term."""
        
        lineage = spark.sql(f"""
            SELECT 
                l.database_name,
                l.table_name,
                l.column_name,
                l.full_path,
                l.confidence_score,
                l.linkage_type,
                l.verified,
                t.term_name,
                t.definition
            FROM metadata.glossary_term_linkages l
            JOIN metadata.glossary_terms t ON l.term_id = t.term_id
            WHERE l.term_id = '{term_id}'
            ORDER BY l.database_name, l.table_name, l.column_name
        """)
        
        return lineage

print("\n✅ BusinessGlossaryManager class loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 3: Policy Engine — Automated Governance                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

class GovernancePolicyEngine:
    """
    Automated policy engine for metadata governance.
    Scans all datasets and enforces standards.
    """
    
    def __init__(self):
        self.glossary_manager = BusinessGlossaryManager()
    
    def create_policy(
        self,
        policy_name: str,
        policy_type: str,
        description: str,
        severity: str = "high",
        enabled: bool = True,
        scope: str = "all",
        validation_query: str = None
    ) -> str:
        """
        Create a governance policy.
        
        Args:
            policy_name: Policy name
            policy_type: ddl_comment, business_definition, naming_convention
            description: What the policy checks
            severity: critical, high, medium, low
            enabled: Is policy active
            scope: all, database, table_pattern
            validation_query: SQL query to detect violations
            
        Returns:
            policy_id
        """
        from notebookutils import mssparkutils
        
        policy_id = f"POLICY_{datetime.now().strftime('%Y%m%d%H%M%S')}_{hash(policy_name) % 10000}"
        current_user = mssparkutils.runtime.context.get("userEmail", "system")
        
        policy_record = {
            "policy_id": policy_id,
            "policy_name": policy_name,
            "policy_type": policy_type,
            "description": description,
            "severity": severity,
            "enabled": enabled,
            "auto_remediate": False,
            "scope": scope,
            "scope_filter": None,
            "validation_query": validation_query,
            "created_by": current_user,
            "created_timestamp": datetime.now(),
            "last_run_timestamp": None
        }
        
        spark.createDataFrame([policy_record]).write.format("delta") \
            .mode("append") \
            .saveAsTable("metadata.governance_policies")
        
        print(f"✅ Policy created: {policy_name} (ID: {policy_id})")
        return policy_id
    
    def scan_ddl_comments(self, database_name: str = None) -> int:
        """
        Scan for tables/columns without DDL comments.
        Returns count of violations found.
        """
        print(f"\n🔍 Scanning DDL comments{' for ' + database_name if database_name else ' (all databases)'}...")
        
        # Get policy
        policy = spark.sql("""
            SELECT policy_id, policy_name, severity
            FROM metadata.governance_policies
            WHERE policy_type = 'ddl_comment'
              AND enabled = true
            LIMIT 1
        """).collect()
        
        if not policy:
            print("⚠️  No DDL comment policy found. Creating default...")
            policy_id = self.create_policy(
                policy_name="DDL Comments Required",
                policy_type="ddl_comment",
                description="All tables and columns must have DDL comments",
                severity="high"
            )
        else:
            policy_id = policy[0]["policy_id"]
            policy_name = policy[0]["policy_name"]
        
        # Get all databases to scan
        databases = ["bronze", "silver", "gold", "metadata"]
        if database_name:
            databases = [database_name]
        
        violations = []
        
        for db in databases:
            try:
                # Get all tables in database
                tables = spark.sql(f"SHOW TABLES IN {db}").collect()
                
                for table_row in tables:
                    table_name = table_row["tableName"]
                    
                    # Get table description
                    table_desc = spark.sql(f"DESCRIBE EXTENDED {db}.{table_name}") \
                        .filter(col("col_name") == "Comment") \
                        .collect()
                    
                    has_table_comment = len(table_desc) > 0 and table_desc[0]["data_type"] not in [None, "NULL", ""]
                    
                    if not has_table_comment:
                        # Table violation
                        violations.append({
                            "violation_id": f"VIO_{datetime.now().strftime('%Y%m%d%H%M%S')}_{hash(table_name) % 10000}",
                            "policy_id": policy_id,
                            "policy_name": policy_name,
                            "severity": "high",
                            "object_type": "table",
                            "database_name": db,
                            "schema_name": "dbo",
                            "table_name": table_name,
                            "column_name": None,
                            "full_path": f"{db}.{table_name}",
                            "violation_details": "Table has no DDL comment",
                            "detected_timestamp": datetime.now(),
                            "status": "new",
                            "remediation_suggestion": f"ALTER TABLE {db}.{table_name} SET TBLPROPERTIES ('comment' = 'Add description here');",
                            "remediated_by": None,
                            "remediated_timestamp": None,
                            "waiver_reason": None,
                            "waived_by": None
                        })
                    
                    # Get column descriptions
                    columns = spark.sql(f"DESCRIBE {db}.{table_name}").collect()
                    
                    for col_row in columns:
                        col_name = col_row["col_name"]
                        col_comment = col_row["comment"]
                        
                        # Skip metadata columns
                        if col_name.startswith("_") or col_name in ["col_name", "data_type", "comment"]:
                            continue
                        
                        if not col_comment or col_comment in ["NULL", ""]:
                            # Column violation
                            violations.append({
                                "violation_id": f"VIO_{datetime.now().strftime('%Y%m%d%H%M%S')}_{hash(col_name) % 10000}",
                                "policy_id": policy_id,
                                "policy_name": policy_name,
                                "severity": "medium",
                                "object_type": "column",
                                "database_name": db,
                                "schema_name": "dbo",
                                "table_name": table_name,
                                "column_name": col_name,
                                "full_path": f"{db}.{table_name}.{col_name}",
                                "violation_details": "Column has no DDL comment",
                                "detected_timestamp": datetime.now(),
                                "status": "new",
                                "remediation_suggestion": f"Add comment via ALTER TABLE or update schema",
                                "remediated_by": None,
                                "remediated_timestamp": None,
                                "waiver_reason": None,
                                "waived_by": None
                            })
            
            except Exception as e:
                print(f"   ⚠️  Error scanning database {db}: {e}")
        
        # Save violations
        if violations:
            spark.createDataFrame(violations).write.format("delta") \
                .mode("append") \
                .saveAsTable("metadata.governance_policy_violations")
        
        print(f"\n   ✅ Scan complete: {len(violations)} violations found")
        return len(violations)
    
    def scan_business_definitions(self, database_name: str = None) -> int:
        """
        Scan for columns without business glossary linkage.
        Returns count of violations found.
        """
        print(f"\n🔍 Scanning business definitions{' for ' + database_name if database_name else ' (all databases)'}...")
        
        # Get policy
        policy = spark.sql("""
            SELECT policy_id, policy_name, severity
            FROM metadata.governance_policies
            WHERE policy_type = 'business_definition'
              AND enabled = true
            LIMIT 1
        """).collect()
        
        if not policy:
            print("⚠️  No business definition policy found. Creating default...")
            policy_id = self.create_policy(
                policy_name="Business Definitions Required",
                policy_type="business_definition",
                description="All columns must be linked to business glossary terms",
                severity="high"
            )
        else:
            policy_id = policy[0]["policy_id"]
            policy_name = policy[0]["policy_name"]
        
        # Get all linkages
        linkages = spark.sql("""
            SELECT full_path 
            FROM metadata.glossary_term_linkages
            WHERE verified = true
        """).collect()
        
        linked_paths = {row["full_path"] for row in linkages}
        
        # Scan databases
        databases = ["bronze", "silver", "gold"]
        if database_name:
            databases = [database_name]
        
        violations = []
        
        for db in databases:
            try:
                tables = spark.sql(f"SHOW TABLES IN {db}").collect()
                
                for table_row in tables:
                    table_name = table_row["tableName"]
                    
                    # Get columns
                    columns = spark.sql(f"DESCRIBE {db}.{table_name}").collect()
                    
                    for col_row in columns:
                        col_name = col_row["col_name"]
                        
                        # Skip metadata columns
                        if col_name.startswith("_") or col_name in ["col_name", "data_type", "comment"]:
                            continue
                        
                        full_path = f"{db}.{table_name}.{col_name}"
                        
                        if full_path not in linked_paths:
                            # No business definition
                            violations.append({
                                "violation_id": f"VIO_{datetime.now().strftime('%Y%m%d%H%M%S')}_{hash(full_path) % 10000}",
                                "policy_id": policy_id,
                                "policy_name": policy_name,
                                "severity": "medium",
                                "object_type": "column",
                                "database_name": db,
                                "schema_name": "dbo",
                                "table_name": table_name,
                                "column_name": col_name,
                                "full_path": full_path,
                                "violation_details": "Column not linked to business glossary term",
                                "detected_timestamp": datetime.now(),
                                "status": "new",
                                "remediation_suggestion": f"Link column to appropriate glossary term or create new term",
                                "remediated_by": None,
                                "remediated_timestamp": None,
                                "waiver_reason": None,
                                "waived_by": None
                            })
            
            except Exception as e:
                print(f"   ⚠️  Error scanning database {db}: {e}")
        
        # Save violations
        if violations:
            spark.createDataFrame(violations).write.format("delta") \
                .mode("append") \
                .saveAsTable("metadata.governance_policy_violations")
        
        print(f"\n   ✅ Scan complete: {len(violations)} violations found")
        return len(violations)
    
    def run_all_policies(self) -> Dict:
        """Run all enabled policies."""
        print("\n" + "=" * 80)
        print("🛡️  RUNNING ALL GOVERNANCE POLICIES")
        print("=" * 80)
        
        results = {
            "ddl_comments": 0,
            "business_definitions": 0,
            "total_violations": 0
        }
        
        # Run DDL comment scan
        results["ddl_comments"] = self.scan_ddl_comments()
        
        # Run business definition scan
        results["business_definitions"] = self.scan_business_definitions()
        
        # Calculate totals
        results["total_violations"] = results["ddl_comments"] + results["business_definitions"]
        
        print("\n" + "=" * 80)
        print("📊 POLICY SCAN SUMMARY")
        print("=" * 80)
        print(f"   DDL Comment Violations: {results['ddl_comments']}")
        print(f"   Business Definition Violations: {results['business_definitions']}")
        print(f"   Total Violations: {results['total_violations']}")
        print("=" * 80)
        
        return results
    
    def generate_compliance_scorecard(self) -> DataFrame:
        """Generate compliance scorecard for all tables."""
        print("\n📊 Generating compliance scorecard...")
        
        scorecard_data = []
        
        databases = ["bronze", "silver", "gold"]
        
        for db in databases:
            try:
                tables = spark.sql(f"SHOW TABLES IN {db}").collect()
                
                for table_row in tables:
                    table_name = table_row["tableName"]
                    
                    # Get column count
                    columns = spark.sql(f"DESCRIBE {db}.{table_name}").collect()
                    total_columns = len([c for c in columns if not c["col_name"].startswith("_")])
                    
                    # Count columns with comments
                    cols_with_comments = len([c for c in columns 
                                             if c["comment"] and c["comment"] not in ["NULL", ""] 
                                             and not c["col_name"].startswith("_")])
                    
                    # Count columns with glossary terms
                    linkages = spark.sql(f"""
                        SELECT COUNT(*) as cnt
                        FROM metadata.glossary_term_linkages
                        WHERE database_name = '{db}'
                          AND table_name = '{table_name}'
                          AND verified = true
                    """).collect()[0]["cnt"]
                    
                    # Count violations
                    violations = spark.sql(f"""
                        SELECT 
                            severity,
                            COUNT(*) as cnt
                        FROM metadata.governance_policy_violations
                        WHERE database_name = '{db}'
                          AND table_name = '{table_name}'
                          AND status = 'new'
                        GROUP BY severity
                    """).collect()
                    
                    vio_dict = {v["severity"]: v["cnt"] for v in violations}
                    
                    # Calculate compliance score
                    compliance_score = 0.0
                    if total_columns > 0:
                        comment_score = cols_with_comments / total_columns * 40
                        glossary_score = linkages / total_columns * 60
                        compliance_score = comment_score + glossary_score
                    
                    scorecard_data.append({
                        "scorecard_id": f"SCORE_{datetime.now().strftime('%Y%m%d%H%M%S')}_{hash(table_name) % 10000}",
                        "database_name": db,
                        "table_name": table_name,
                        "total_columns": total_columns,
                        "columns_with_comments": cols_with_comments,
                        "columns_with_business_definitions": linkages,
                        "columns_with_glossary_terms": linkages,
                        "compliance_score": round(compliance_score, 2),
                        "last_scanned_timestamp": datetime.now(),
                        "violations_critical": vio_dict.get("critical", 0),
                        "violations_high": vio_dict.get("high", 0),
                        "violations_medium": vio_dict.get("medium", 0),
                        "violations_low": vio_dict.get("low", 0)
                    })
            
            except Exception as e:
                print(f"   ⚠️  Error processing database {db}: {e}")
        
        # Save scorecard
        if scorecard_data:
            scorecard_df = spark.createDataFrame(scorecard_data)
            scorecard_df.write.format("delta") \
                .mode("overwrite") \
                .saveAsTable("metadata.metadata_compliance_scorecard")
            
            print(f"   ✅ Scorecard generated: {len(scorecard_data)} tables")
            
            return scorecard_df
        
        return None

print("\n✅ GovernancePolicyEngine class loaded")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 4: Seed Insurance Business Glossary                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("🌱 SEEDING INSURANCE BUSINESS GLOSSARY")
print("=" * 80)

glossary = BusinessGlossaryManager()

# Insurance-specific business terms
insurance_terms = [
    {
        "term_name": "premium_amount",
        "definition": "The amount of money paid for insurance coverage",
        "business_definition": "Regular payment made by policyholder to insurance company for coverage",
        "technical_definition": "DECIMAL(18,2) monetary value representing premium payment",
        "category": "Financial",
        "domain": "Policy",
        "tags": ["finance", "policy", "payment"]
    },
    {
        "term_name": "deductible_amount",
        "definition": "Amount policyholder must pay before insurance coverage begins",
        "business_definition": "Out-of-pocket expense before insurance pays",
        "technical_definition": "DECIMAL(18,2) threshold amount for claim coverage",
        "category": "Financial",
        "domain": "Claims",
        "tags": ["finance", "claims", "coverage"]
    },
    {
        "term_name": "policy_number",
        "definition": "Unique identifier for insurance policy",
        "business_definition": "Unique policy contract identifier issued to customer",
        "technical_definition": "VARCHAR(50) primary key for policy entity",
        "category": "Identifier",
        "domain": "Policy",
        "tags": ["identifier", "policy", "key"]
    },
    {
        "term_name": "claim_amount",
        "definition": "Total amount requested in insurance claim",
        "business_definition": "Dollar value of customer claim for covered incident",
        "technical_definition": "DECIMAL(18,2) total claim valuation",
        "category": "Financial",
        "domain": "Claims",
        "tags": ["finance", "claims", "payment"]
    },
    {
        "term_name": "loss_ratio",
        "definition": "Ratio of claims paid to premiums earned",
        "business_definition": "Key profitability metric: (Claims Paid / Premiums Earned) * 100",
        "technical_definition": "DECIMAL(5,2) calculated KPI metric",
        "category": "Metric",
        "domain": "Finance",
        "tags": ["kpi", "finance", "ratio"]
    },
    {
        "term_name": "policy_effective_date",
        "definition": "Date when insurance coverage begins",
        "business_definition": "Start date of policy coverage period",
        "technical_definition": "DATE field marking coverage start",
        "category": "Temporal",
        "domain": "Policy",
        "tags": ["date", "policy", "coverage"]
    },
    {
        "term_name": "fnol_date",
        "definition": "First Notice of Loss date - when claim is first reported",
        "business_definition": "Date policyholder notifies insurer of covered incident",
        "technical_definition": "TIMESTAMP of initial claim report",
        "category": "Temporal",
        "domain": "Claims",
        "tags": ["date", "claims", "timeline"]
    },
    {
        "term_name": "customer_segment",
        "definition": "Classification of customer by business characteristics",
        "business_definition": "Customer grouping for targeted marketing and risk assessment",
        "technical_definition": "VARCHAR(50) categorical dimension",
        "category": "Classification",
        "domain": "Customer",
        "tags": ["customer", "segmentation", "marketing"]
    }
]

# Create terms
created_terms = []
for term in insurance_terms:
    try:
        term_id = glossary.create_term(
            term_name=term["term_name"],
            definition=term["definition"],
            category=term["category"],
            domain=term["domain"],
            business_definition=term["business_definition"],
            technical_definition=term["technical_definition"],
            tags=term["tags"]
        )
        
        # Auto-approve for demo
        glossary.approve_term(term_id)
        
        created_terms.append({"term_id": term_id, "term_name": term["term_name"]})
        
    except Exception as e:
        print(f"   ⚠️  Failed to create term {term['term_name']}: {e}")

print(f"\n✅ Created {len(created_terms)} glossary terms")
print("=" * 80)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 5: Auto-Link Glossary Terms to Physical Columns            ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("🔗 AUTO-LINKING GLOSSARY TERMS TO PHYSICAL COLUMNS")
print("=" * 80)

# Get all published terms
terms_df = spark.sql("""
    SELECT term_id, term_name
    FROM metadata.glossary_terms
    WHERE status = 'published'
""")

# Scan databases for matching column names
databases = ["bronze", "silver", "gold"]
linkage_count = 0

for term_row in terms_df.collect():
    term_id = term_row["term_id"]
    term_name = term_row["term_name"]
    
    print(f"\nLinking term: {term_name}")
    
    for db in databases:
        try:
            tables = spark.sql(f"SHOW TABLES IN {db}").collect()
            
            for table_row in tables:
                table_name = table_row["tableName"]
                
                # Get columns
                columns = spark.sql(f"DESCRIBE {db}.{table_name}").collect()
                
                for col_row in columns:
                    col_name = col_row["col_name"]
                    
                    # Check for exact or fuzzy match
                    if col_name.lower() == term_name.lower():
                        # Exact match - high confidence
                        glossary.link_term_to_column(
                            term_id=term_id,
                            database_name=db,
                            table_name=table_name,
                            column_name=col_name,
                            confidence=1.0,
                            linkage_type="auto_approved"
                        )
                        linkage_count += 1
                        print(f"   ✅ Linked: {db}.{table_name}.{col_name} (exact match)")
                    
                    elif term_name.lower() in col_name.lower() or col_name.lower() in term_name.lower():
                        # Partial match - medium confidence
                        glossary.link_term_to_column(
                            term_id=term_id,
                            database_name=db,
                            table_name=table_name,
                            column_name=col_name,
                            confidence=0.7,
                            linkage_type="auto_suggested"
                        )
                        linkage_count += 1
                        print(f"   ⚠️  Suggested: {db}.{table_name}.{col_name} (partial match - needs review)")
        
        except Exception as e:
            print(f"   ⚠️  Error scanning {db}: {e}")

print(f"\n✅ Created {linkage_count} term linkages")
print("=" * 80)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 6: Run Policy Engine & Generate Reports                    ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Initialize policy engine
policy_engine = GovernancePolicyEngine()

# Run all policies
scan_results = policy_engine.run_all_policies()

# Generate compliance scorecard
scorecard = policy_engine.generate_compliance_scorecard()

# Display scorecard
if scorecard:
    print("\n" + "=" * 80)
    print("📊 METADATA COMPLIANCE SCORECARD")
    print("=" * 80)
    
    display(scorecard.orderBy(desc("compliance_score")))

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SECTION 7: Violation Reports & Dashboards                          ║
# ╚══════════════════════════════════════════════════════════════════════╝

print("\n" + "=" * 80)
print("📋 GOVERNANCE VIOLATION REPORTS")
print("=" * 80)

# === Top Violations by Severity ===
print("\n🚨 Top Violations by Severity")

violations_by_severity = spark.sql("""
    SELECT 
        severity,
        COUNT(*) as violation_count,
        COUNT(DISTINCT database_name) as affected_databases,
        COUNT(DISTINCT table_name) as affected_tables
    FROM metadata.governance_policy_violations
    WHERE status = 'new'
    GROUP BY severity
    ORDER BY 
        CASE severity
            WHEN 'critical' THEN 1
            WHEN 'high' THEN 2
            WHEN 'medium' THEN 3
            WHEN 'low' THEN 4
        END
""")

display(violations_by_severity)

# === Tables with Most Violations ===
print("\n⚠️  Tables with Most Violations")

tables_with_violations = spark.sql("""
    SELECT 
        database_name,
        table_name,
        COUNT(*) as violation_count,
        SUM(CASE WHEN severity = 'critical' THEN 1 ELSE 0 END) as critical_count,
        SUM(CASE WHEN severity = 'high' THEN 1 ELSE 0 END) as high_count,
        SUM(CASE WHEN severity = 'medium' THEN 1 ELSE 0 END) as medium_count
    FROM metadata.governance_policy_violations
    WHERE status = 'new'
    GROUP BY database_name, table_name
    ORDER BY violation_count DESC
    LIMIT 10
""")

display(tables_with_violations)

# === Glossary Coverage Summary ===
print("\n📚 Business Glossary Coverage")

glossary_coverage = spark.sql("""
    SELECT 
        database_name,
        COUNT(DISTINCT table_name) as total_tables,
        COUNT(DISTINCT CASE WHEN columns_with_glossary_terms > 0 THEN table_name END) as tables_with_glossary,
        ROUND(AVG(compliance_score), 2) as avg_compliance_score,
        SUM(total_columns) as total_columns,
        SUM(columns_with_comments) as columns_with_comments,
        SUM(columns_with_glossary_terms) as columns_with_glossary
    FROM metadata.metadata_compliance_scorecard
    GROUP BY database_name
    ORDER BY avg_compliance_score DESC
""")

display(glossary_coverage)

# === Recent Glossary Terms ===
print("\n📖 Recently Published Glossary Terms")

recent_terms = spark.sql("""
    SELECT 
        term_name,
        display_name,
        definition,
        category,
        domain,
        approved_by,
        approved_timestamp
    FROM metadata.glossary_terms
    WHERE status = 'published'
    ORDER BY approved_timestamp DESC
    LIMIT 10
""")

display(recent_terms)

print("\n" + "=" * 80)

## 🚀 Usage Examples

### Create a New Business Term

```python
glossary = BusinessGlossaryManager()

term_id = glossary.create_term(
    term_name="net_written_premium",
    definition="Total premiums written minus cancellations and returns",
    business_definition="Revenue metric representing actual premium income after adjustments",
    technical_definition="DECIMAL(18,2) calculated as: gross_premium - cancellations - returns",
    category="Financial",
    domain="Finance",
    tags=["revenue", "metrics", "finance"]
)

# Approve the term
glossary.approve_term(term_id)

# Link to physical column
glossary.link_term_to_column(
    term_id=term_id,
    database_name="gold",
    table_name="fact_premiums",
    column_name="net_written_premium"
)
```

### Run Policy Scan on Specific Database

```python
policy_engine = GovernancePolicyEngine()

# Scan Gold layer only
policy_engine.scan_ddl_comments("gold")
policy_engine.scan_business_definitions("gold")
```

### Search Glossary

```python
# Search for terms
results = glossary.search_terms("premium")
display(results)

# Get term lineage
lineage = glossary.get_term_lineage(term_id)
display(lineage)
```

---

## 📊 Integration with Other Notebooks

### Notebook 40 (Data Quality Framework)
```python
# Add glossary compliance to DQ checks
def check_metadata_quality():
    scorecard = spark.table("metadata.metadata_compliance_scorecard")
    low_compliance = scorecard.filter(col("compliance_score") < 50)
    
    if low_compliance.count() > 0:
        send_alert("Low metadata compliance detected!")
```

### Notebook 45 (Operational Monitoring)
```python
# Add to health checks
def check_governance_health():
    violations = spark.sql("""
        SELECT COUNT(*) as critical_violations
        FROM metadata.governance_policy_violations
        WHERE severity = 'critical' AND status = 'new'
    """).collect()[0]["critical_violations"]
    
    if critical_violations > 0:
        send_alert(f"🚨 {critical_violations} critical governance violations!")
```

### Notebook 90 (Central Dashboard)
```python
# Add governance metrics
governance_metrics = spark.sql("""
    SELECT 
        COUNT(*) as total_terms,
        SUM(CASE WHEN status = 'published' THEN 1 ELSE 0 END) as published_terms,
        COUNT(DISTINCT domain) as domains_covered
    FROM metadata.glossary_terms
""")
```

---

## ⏰ Scheduled Execution

**Daily Governance Scan (Fabric Pipeline):**

```json
{
  "name": "Daily Governance Scan",
  "activities": [
    {
      "type": "SynapseNotebook",
      "notebook": "67_business_glossary_policy_engine",
      "section": "6"
    }
  ],
  "trigger": {
    "type": "Schedule",
    "recurrence": {
      "frequency": "Day",
      "interval": 1,
      "startTime": "02:00:00"
    }
  }
}
```

---

## ✅ Summary

**This notebook provides:**

✅ **Business Glossary Management** — Self-contained, no Purview  
✅ **Term lifecycle** — Draft → Review → Approved → Published  
✅ **Auto-linking** — Match terms to physical columns  
✅ **Policy Engine** — Automated governance enforcement  
✅ **DDL comment validation** — Flag uncommented tables/columns  
✅ **Business definition validation** — Flag unmapped columns  
✅ **Compliance scorecard** — Per-table metadata quality scores  
✅ **Violation tracking** — Full audit trail  
✅ **Auto-remediation** — Suggest fixes  
✅ **Integration ready** — Works with DQ and monitoring  

**Enterprise data governance without external dependencies!** 📚